<a href="https://colab.research.google.com/github/BrundaSreedhar/LLM-Fine-Tuning/blob/main/Supervised_Fine_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install torch datasets transformers trl==0.14.0

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 313.9/313.9 kB 6.4 MB/s eta 0:00:00


In [2]:
import torch
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import TrainingArguments, AutoTokenizer, AutoModelForCausalLM
from trl import SFTTrainer, DataCollatorForCompletionOnlyLM, SFTConfig

In [19]:
def generate_responses(model, tokenizer, user_message, system_message=None,
                       max_new_tokens=250):
    # Format chat using tokenizer's chat template
    messages = []
    if system_message:
        messages.append({"role": "system", "content": system_message})

    # We assume the data are all single-turn conversation
    messages.append({"role": "user", "content": user_message})

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    # Recommended to use vllm, sglang or TensorRT
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    generated_ids = outputs[0][input_len:]
    response = tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return response

In [4]:
system_message = f"""You are a helpful customer support agent.
When responding:
- Acknowledge the customer's issue briefly in one sentence
- Give clear, concise steps to resolve the issue
- Never invent specific UI elements like button names or tab names you don't know exist
- If you cannot resolve the issue, ask the customer to contact the support team
- Keep responses under 5 sentences or steps
- Always complete your response fully, never trail off"""


In [5]:
def test_model_with_questions(model, tokenizer, questions,
                              system_message=None, title="Model Output"):
    print(f"\n=== {title} ===")
    for i, question in enumerate(questions, 1):
        response = generate_responses(model, tokenizer, question,
                                      system_message)
        print(f"\nModel Input {i}:\n{question}\nModel Output {i}:\n{response}\n")


In [6]:
def load_model_and_tokenizer(model_name, use_gpu = False):

    # Load base model and tokenizer
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForCausalLM.from_pretrained(model_name)

    if use_gpu:
        model.to("cuda")

    if not tokenizer.chat_template:
        tokenizer.chat_template = """{% for message in messages %}
                {% if message['role'] == 'system' %}System: {{ message['content'] }}\n
                {% elif message['role'] == 'user' %}User: {{ message['content'] }}\n
                {% elif message['role'] == 'assistant' %}Assistant: {{ message['content'] }} <|endoftext|>
                {% endif %}
                {% endfor %}"""

    # Tokenizer config
    if not tokenizer.pad_token:
        tokenizer.pad_token = tokenizer.eos_token

    return model, tokenizer

In [7]:
def display_dataset(dataset, num_rows=3, max_colwidth=100):
    rows = []

    for i in range(min(num_rows, len(dataset))):
        example = dataset[i]

        # Handle both common dataset formats
        if 'messages' in example:
            # Chat/conversation format (your current approach)
            user_msg = next(
                (m['content'] for m in example['messages'] if m['role'] == 'user'),
                'N/A'
            )
            assistant_msg = next(
                (m['content'] for m in example['messages'] if m['role'] == 'assistant'),
                'N/A'
            )
        elif 'instruction' in example:
            # Instruction/response format (Alpaca-style)
            user_msg = example['instruction']
            assistant_msg = example.get('output') or example.get('response', 'N/A')
        else:
            # Fallback: just show raw keys
            user_msg = str(example)
            assistant_msg = 'Unknown format'

        rows.append({
            'Index': i,
            'User Prompt': user_msg[:max_colwidth] + '...' if len(user_msg) > max_colwidth else user_msg,
            'Assistant Response': assistant_msg[:max_colwidth] + '...' if len(assistant_msg) > max_colwidth else assistant_msg
        })

    df = pd.DataFrame(rows).set_index('Index')
    pd.set_option('display.max_colwidth', None)
    display(df)

Load base model & test on simple questions

In [8]:
USE_GPU = torch.cuda.is_available()

questions = [
    "I was charged twice this month, how do I get a refund?",
    "I forgot my password and the reset email is not arriving.",
    "How do I cancel my subscription before the next billing cycle?"
]

In [9]:
mistral_model, mistral_tokenizer = load_model_and_tokenizer("TinyLlama/TinyLlama-1.1B-Chat-v1.0", USE_GPU)

# Explicitly disable gradient checkpointing on the model to avoid 'torch.xla' error
if hasattr(mistral_model, "gradient_checkpointing_disable"):
    mistral_model.gradient_checkpointing_disable()

config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [10]:
test_model_with_questions(mistral_model, mistral_tokenizer, questions, system_message,
                          title="TinyLlama Model Output (Before SFT)")


=== TinyLlama Model Output (Before SFT) ===

Model Input 1:
I was charged twice this month, how do I get a refund?
Model Output 1:
To get a refund for charges twice this month, you can follow these steps:

1. Contact the company that charged you twice. You can do this by calling their customer support number or visiting their website and filling out a form to request a refund.

2. Provide them with your account information, including your billing address, payment method, and order number.

3. Explain the situation to the customer service representative. Be clear and concise about


Model Input 2:
I forgot my password and the reset email is not arriving.
Model Output 2:
I apologize for the inconvenience. If you forgot your password and the reset email is not arriving, please check your spam or junk folder for the email. If you still cannot find it, please contact our support team for further assistance. We are here to help you.


Model Input 3:
How do I cancel my subscription before th

In [11]:
train_dataset = load_dataset("bitext/Bitext-customer-support-llm-chatbot-training-dataset")["train"]
if not USE_GPU:
    train_dataset=train_dataset.select(range(100))
else: # If USE_GPU is True, select a larger subset than 100 for faster experimentation
    train_dataset=train_dataset.select(range(5000)) # Using 5000 examples for faster training

display_dataset(train_dataset)

README.md: 0.00B [00:00, ?B/s]

Bitext_Sample_Customer_Support_Training_(…):   0%|          | 0.00/19.2M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/26872 [00:00<?, ? examples/s]

,User Prompt,Assistant Response
Index,,
0,question about cancelling order {{Order Number}},"I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to prov..."
1,i have a question about cancelling oorder {{Order Number}},I've been informed that you have a question about canceling order {{Order Number}}. I'm here to assi...
2,i need help cancelling puchase {{Order Number}},I can sense that you're seeking assistance with canceling your purchase with the purchase number {{O...


Stage 3 — QLoRA SFT on TinyLLama (~30 mins on a T4)



In [12]:
# SFTTrainer config
sft_config = SFTConfig(
    learning_rate=8e-5, # Learning rate for training.
    num_train_epochs=1, #  Set the number of epochs to train the model.
    per_device_train_batch_size=1, # Batch size for each device (e.g., GPU) during training.
    gradient_accumulation_steps=8, # Number of steps before performing a backward/update pass to accumulate gradients.
    gradient_checkpointing=False, # Disable gradient checkpointing when not using GPU.
    logging_steps=2,  # Frequency of logging training progress (log every 2 steps).
    packing=True, # Enable data packing for more efficient GPU utilization
)

In [13]:
print(train_dataset[0].keys())  # see exactly what's there
print(train_dataset[0])         # inspect a full example

dict_keys(['flags', 'instruction', 'category', 'intent', 'response'])
{'flags': 'B', 'instruction': 'question about cancelling order {{Order Number}}', 'category': 'ORDER', 'intent': 'cancel_order', 'response': "I've understood you have a question regarding canceling order {{Order Number}}, and I'm here to provide you with the information you need. Please go ahead and ask your question, and I'll do my best to assist you."}


In [14]:
def formatting_function(example):
    # SFTTrainer can pass batched examples — handle both cases
    if isinstance(example["instruction"], list):
        # Batched: process each example individually
        return [
            mistral_tokenizer.apply_chat_template(
                [
                    {"role": "user",      "content": instr},
                    {"role": "assistant", "content": resp}
                ],
                tokenize=False,
                add_generation_prompt=False
            ) + mistral_tokenizer.eos_token
            for instr, resp in zip(example["instruction"], example["response"])
        ]
    else:
        # Single example
        messages = [
            {"role": "user",      "content": example["instruction"]},
            {"role": "assistant", "content": example["response"]}
        ]
        return mistral_tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        ) + mistral_tokenizer.eos_token

In [15]:
sft_trainer = SFTTrainer(
    model=mistral_model,
    args=sft_config,
    train_dataset=train_dataset,
    processing_class=mistral_tokenizer,
    formatting_func=formatting_function,
)
sft_trainer.train()

Generating train split: 0 examples [00:00, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
2,1.699901
4,1.303842
6,0.947928
8,0.848502
10,0.718609
12,0.673404
14,0.646293
16,0.643293
18,0.628229
20,0.617552


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=93, training_loss=0.6089777148539021, metrics={'train_runtime': 2547.8934, 'train_samples_per_second': 0.292, 'train_steps_per_second': 0.037, 'total_flos': 4722540756860928.0, 'train_loss': 0.6089777148539021, 'epoch': 1.0})

In [16]:
from huggingface_hub import login
HF_TOKEN = "your_hf_token"
login(HF_TOKEN)  # get from huggingface.co/settings/tokens

# Save
sft_trainer.model.push_to_hub("iambrundy/tinyllama-customer-support-v1")
mistral_tokenizer.push_to_hub("iambrundy/tinyllama-customer-support-v1")


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...e06mgjg/model.safetensors:   0%|          |  607kB / 2.20GB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/iambrundy/tinyllama-customer-support-v1/commit/07b6d086d2bac50528ae2d7d25b85560c85ca266', commit_message='Upload tokenizer', commit_description='', oid='07b6d086d2bac50528ae2d7d25b85560c85ca266', pr_url=None, repo_url=RepoUrl('https://huggingface.co/iambrundy/tinyllama-customer-support-v1', endpoint='https://huggingface.co', repo_type='model', repo_id='iambrundy/tinyllama-customer-support-v1'), pr_revision=None, pr_num=None)

In [17]:
# Load later from anywhere
from transformers import AutoModelForCausalLM, AutoTokenizer
model = AutoModelForCausalLM.from_pretrained("iambrundy/tinyllama-customer-support-v1")
tokenizer = AutoTokenizer.from_pretrained("iambrundy/tinyllama-customer-support-v1")

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/133 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

chat_template.jinja:   0%|          | 0.00/410 [00:00<?, ?B/s]

In [20]:
if not USE_GPU: # move model to CPU when GPU isn’t requested
    model.to("cpu")
test_model_with_questions(model, tokenizer, questions, system_message,
                          title="Base Model (After SFT) Output")


=== Base Model (After SFT) Output ===

Model Input 1:
I was charged twice this month, how do I get a refund?
Model Output 1:
I'm sorry to hear that you're facing charges twice this month. To get a refund, please follow these steps:

1. Log in to your account on our website.
2. Navigate to the "Account" or "Billing" section.
3. Look for the "Refunds" or "Cancellation Fees" tab.
4. Click on it to view the details of the charges.
5. If you're eligible for a refund, you can either contact our customer support team or reach out to your bank for assistance.

If you encounter any difficulties or have further questions, please don't hesitate to reach out to our customer support team. We're here to help you every step of the way.


Model Input 2:
I forgot my password and the reset email is not arriving.
Model Output 2:
I'm sorry to hear that you're experiencing difficulties with your password and the reset email. I'll do my best to assist you in resolving this issue. Could you please provide m

In [21]:
test_model_with_questions(
    mistral_model,
    mistral_tokenizer,
    questions,
    system_message=system_message,
    title="TinyLlama Model Output (After SFT + System Prompt)"
)


=== TinyLlama Model Output (After SFT + System Prompt) ===

Model Input 1:
I was charged twice this month, how do I get a refund?
Model Output 1:
I'm sorry to hear that you're facing charges twice this month. To get a refund, please follow these steps:

1. Log in to your account on our website.
2. Navigate to the "Account" or "Billing" section.
3. Look for the "Refunds" or "Cancellation Fees" tab.
4. Click on it to view the details of the charges.
5. If you're eligible for a refund, you can either contact our customer support team or reach out to your bank for assistance.

If you encounter any difficulties or have further questions, please don't hesitate to reach out to our customer support team. We're here to help you every step of the way.


Model Input 2:
I forgot my password and the reset email is not arriving.
Model Output 2:
I'm sorry to hear that you're having trouble with your password and the reset email. Don't worry, I'm here to help you with that. To resolve the issue, plea

Stage 4 — Evals
